In [ ]:
!pip install 'lighteval[vllm]'
!pip install transformers -U

  Preparing metadata (setup.py) ... done
INFO: pip is looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
INFO: pip is still looking at multiple versions of s3fs to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 82.0/82.0 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.5/144.5 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 34.5/34.5 MB 33.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 414.4/414.4 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 821.0/821.0 MB 1.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 169.0/169.0 kB 17.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [ ]:
import lighteval
from lighteval.logging.evaluation_tracker import EvaluationTracker
from lighteval.models.vllm.vllm_model import VLLMModelConfig
from lighteval.pipeline import ParallelismManager, Pipeline, PipelineParameters
from lighteval.utils.imports import is_package_available




In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

def load_model():
    base_model = "FrancescoArno94/SmolLM3-3B-math"
    tokenizer = AutoTokenizer.from_pretrained(base_model, trust_remote_code=True)
    tokenizer.padding_side = "left"

    if tokenizer.pad_token_id is None:
        tokenizer.pad_token_id = tokenizer.eos_token_id

    model = AutoModelForCausalLM.from_pretrained(
        base_model,
        device_map="auto",
        torch_dtype=torch.bfloat16,
        trust_remote_code=True,
    )

    return model, tokenizer

model, tokenizer = load_model()

adapter_config.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/326 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/182 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/242M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/504 [00:00<?, ?it/s]

In [ ]:
evaluation_tracker = EvaluationTracker(
        output_dir="./eval_results",
        save_details=True,
        push_to_hub=True,
        hub_results_org="FrancescoArno94",
    )

pipeline_params = PipelineParameters(
    launcher_type=ParallelismManager.NONE,
    custom_tasks_directory=None,
    max_samples=10,
)



In [ ]:
from lighteval.logging.evaluation_tracker import EvaluationTracker
from lighteval.models.transformers.transformers_model import (
    TransformersModel,
    TransformersModelConfig,
)
from lighteval.pipeline import ParallelismManager, Pipeline, PipelineParameters

In [ ]:
config = TransformersModelConfig(
    model_name="FrancescoArno94/SmolLM3-3B-math",
    batch_size=4,
)

lighteval_model = TransformersModel.from_model(model, config)

# This triggers the FileNotFoundError
pipeline = Pipeline(
    model=lighteval_model,
    pipeline_parameters=pipeline_params,
    evaluation_tracker=evaluation_tracker,
    tasks="lighteval|gsm8k|0",
)



In [ ]:
pipeline.evaluate()

In [ ]:
pipeline.show_results()

| Task  |Version|     Metric     |Value|   |Stderr|
|-------|-------|----------------|----:|---|-----:|
|all    |       |extractive_match|  0.7|±  |0.1528|
|gsm8k:0|       |extractive_match|  0.7|±  |0.1528|



In [ ]:
from google.colab import userdata
hf_token = userdata.get('HF_TOKEN')

In [ ]:
import os
os.environ['HF_TOKEN'] = hf_token


In [ ]:
!hf auth login

User is already logged in.


In [ ]:
pipeline.save_and_push_results()


Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Creating parquet from Arrow format:   0%|          | 0/1 [00:00<?, ?ba/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/hffs-nvcy_q0n          : 100%|##########| 20.2kB / 20.2kB            

...

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  /tmp/hffs-od0b2ksp          : 100%|##########| 38.4kB / 38.4kB            

results_2026-02-04T13-50-32.848574.json:   0%|          | 0.00/4.24k [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

In [ ]:
ls

eval_results/  sample_data/


In [ ]:
?pipeline.save_and_push_results


In [ ]:
ls eval_results

details/  results/


In [ ]:
results = pipeline.get_results()


In [ ]:
results

{'config_general': {'lighteval_sha': '?',
  'num_fewshot_seeds': 1,
  'max_samples': 10,
  'job_id': '0',
  'start_time': 1758.101872877,
  'end_time': 2124.114646524,
  'total_evaluation_time_secondes': '366.0127736469999',
  'model_config': TransformersModelConfig(model_name='FrancescoArno94/SmolLM3-3B-math', generation_parameters=GenerationParameters(num_blocks=None, block_size=None, early_stopping=None, repetition_penalty=None, frequency_penalty=None, length_penalty=None, presence_penalty=None, max_new_tokens=None, min_new_tokens=None, seed=None, stop_tokens=None, temperature=0, top_k=None, min_p=None, top_p=None, truncate_prompt=None, cache_implementation=None, response_format=None), system_prompt=None, cache_dir='~/.cache/huggingface/lighteval', tokenizer=None, subfolder=None, revision='main', batch_size=4, max_length=None, model_loading_kwargs={}, add_special_tokens=True, skip_special_tokens=True, model_parallel=None, dtype=None, device='cuda', trust_remote_code=False, compile=F